# Testing LSST Feature Extraction via reLAISS

This notebook tests code for extracting lightcurve features from LSST alerts via Antares.

**Goal:** Confirm that `extract_lc_and_host_features` produces the same feature columns that Isolation Forest and DiMMAD models were trained on (`constants.lc_features_const`).

**Workflow:**
1. Run extraction script on one LSST object
2. Compare output columns against the training feature lists
3. Check for missing/extra features before passing to models

## 1. Setup & Imports

In [1]:
import os
import sys

import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ── reLAISS path changed from script
import importlib, subprocess

RELAISS_REPO = "/Users/jennakempster-taylor/re-laiss"

if importlib.util.find_spec("relaiss") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", RELAISS_REPO])

# ── Core imports ──────────────────────────────────────────────────────────────
import antares_client
import gdown
from relaiss.features import extract_lc_and_host_features
from relaiss.relaiss import REFERENCE_DIR, download_sfd_files
from relaiss import constants

print("All imports OK")

All imports OK


## 2. What features did we train on?

Before touching the LSST data, record the exact feature lists from your training notebooks.
These are the **ground truth** we need to match.

In [3]:
# These are the same constants your IsoForest and DiMMAD notebooks pulled from
default_lc_features   = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

# The training notebooks dropped *_err columns:
def training_overlap(cols, frame):
    """Mirrors the overlap() function in both training notebooks."""
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

print(f"LC features defined in constants:   {len(default_lc_features)}")
print(f"Host features defined in constants: {len(default_host_features)}")
print(f"Total before filtering:             {len(default_lc_features) + len(default_host_features)}")
print()
print("LC features:")
print(default_lc_features)
print()
print("Host features:")
print(default_host_features)

LC features defined in constants:   25
Host features defined in constants: 18
Total before filtering:             43

LC features:
['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature']

Host features:
['gKronMagCorrected', 'gKronRad', 'gExtNSigma', 'rKronMagCorrected', 'rKronRad', 'rExtNSigma', 'iKronMagCorrected', 'iKronRad', 'iExtNSigma', 'zKronMagCorrected', 'zKronRad', 'zExtNSigma', 'gminusrKronMag', 'rminusiKronMag', 'iminuszKronMag', 'rmomentXX', 'rmomentXY', 'rmomentYY']


## 3. Load your saved model artifacts

Load the fitted imputers, scalers, and feature column lists from training.
This is the safest reference for exactly what columns the models expect.

In [5]:
from joblib import load

# ── Isolation Forest ──────────────────────────────────────────────────────────
# Saved in IsoForestDur.ipynb Cell 32: iforest_artifacts.joblib
# Contains keys: 'iso', 'knn_imp', 'numeric_feature_cols'
ISO_ARTIFACT_PATH = "iforest_artifacts.joblib"  # update path if needed

# ── DiMMAD ────────────────────────────────────────────────────────────────────
# Saved in DimmadTest_cleaned.ipynb: models/dimmad_medmed_relaiss_bundle.joblib
# Contains keys: 'feature_cols', 'imputer', 'scaler', 'model'
DIMMAD_ARTIFACT_PATH = "models/dimmad_medmed_relaiss_bundle.joblib"  # update path if needed

iso_art, dimmad_art = None, None

if Path(ISO_ARTIFACT_PATH).exists():
    iso_art = load(ISO_ARTIFACT_PATH)
    iso_feature_cols = iso_art["numeric_feature_cols"]
    print(f"IsoForest artifact loaded. Expects {len(iso_feature_cols)} features.")
else:
    print(f"WARNING: IsoForest artifact not found at {ISO_ARTIFACT_PATH}")
    print("  -> Will use constants as fallback for feature comparison")
    iso_feature_cols = None

if Path(DIMMAD_ARTIFACT_PATH).exists():
    dimmad_art = load(DIMMAD_ARTIFACT_PATH)
    dimmad_feature_cols = dimmad_art["feature_cols"]
    print(f"DiMMAD artifact loaded.   Expects {len(dimmad_feature_cols)} features.")
else:
    print(f"WARNING: DiMMAD artifact not found at {DIMMAD_ARTIFACT_PATH}")
    dimmad_feature_cols = None

IsoForest artifact loaded. Expects 25 features.
DiMMAD artifact loaded.   Expects 25 features.


## 4. Run the advisor's extraction script

This is your advisor's vibe code, minimally restructured into a callable function
so we can reuse it for multiple LSST objects later.

In [10]:
def extract_features_for_lsst_object(lsst_id: str, base_dir: Path = Path(".")) -> pd.DataFrame:
    """
    Fetch one LSST object from Antares and extract reLAISS features.
    
    Parameters
    ----------
    lsst_id : str
        LSST diaObject ID (e.g. '170019696318349586')
    base_dir : Path
        Working directory for outputs and dust maps.
    
    Returns
    -------
    pd.DataFrame
        One-row DataFrame of reLAISS features for this object.
    """
    sfd_folder       = base_dir / "sfddata-master"
    timeseries_folder = base_dir / "laiss_final" / "timeseries"
    sfd_folder.mkdir(parents=True, exist_ok=True)
    timeseries_folder.mkdir(parents=True, exist_ok=True)

    # Ensure SFD dust maps are present
    download_sfd_files(str(sfd_folder))

    # Ensure reference bank is present (needed for imputation inside reLAISS)
    bank_path = REFERENCE_DIR / "reference_20k.csv" # NEED TO ADD DURATIONS
    if not bank_path.exists():
        print(f"Reference data not found at {bank_path}; downloading...")
        bank_path.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(
            "https://drive.google.com/uc?export=download&id=1uH_03ju50Enb7ZhiduDrmCVTMvTc7bMC",
            str(bank_path),
            quiet=False,
        )

    # Fetch lightcurve from Antares
    print(f"Fetching LSST {lsst_id} from Antares...")
    locus = antares_client.search.get_by_lsst_dia_object_id(lsst_object_id=lsst_id)
    if locus is None:
        raise ValueError(f"Object {lsst_id} not found in Antares")

    lc_df = locus.timeseries.to_pandas()[["ant_passband", "ant_mjd", "ant_mag", "ant_magerr"]]
    
    # NOTE on passband mapping:
    # Your training data used ZTF passbands: 'g' and 'R'.
    # LSST alert passbands from Antares come as 'G' and 'r' (capital G, lowercase r).
    # Previous code maps: 'G' -> 'g'  and  'r' -> 'R'.
    lc_df["ant_passband"] = lc_df["ant_passband"].replace({"G": "g", "r": "R"})
    
    print(f"Lightcurve fetched: {len(lc_df)} alerts, passbands: {lc_df['ant_passband'].unique()}")
    display(lc_df.head())

    # Extract features via reLAISS
    # building_for_AD=True skips host association (PROST), which is what we want
    df_feat = extract_lc_and_host_features(
        ztf_id=lsst_id,                            # used as row label in output
        theorized_lightcurve_df=lc_df,
        path_to_timeseries_folder=str(timeseries_folder),
        path_to_sfd_folder=str(sfd_folder),
        path_to_dataset_bank=str(bank_path),
        building_for_AD=True,
        store_csv=False,
    )

    # Fix object ID column if needed
    if "ztf_object_id" in df_feat.columns:
        df_feat["ztf_object_id"] = lsst_id
    elif df_feat.index.name == "ztf_object_id":
        df_feat = df_feat.reset_index()
        df_feat["ztf_object_id"] = lsst_id

    # Tag with the LSST ID so we can identify rows later
    df_feat["lsst_dia_object_id"] = lsst_id

    print(f"\nExtracted feature row shape: {df_feat.shape}")
    return df_feat


# ── Run on the test object ────────────────────────────────────────────────────
TEST_LSST_ID = "170019696318349586"
df_extracted = extract_features_for_lsst_object(TEST_LSST_ID)

Fetching LSST 170019696318349586 from Antares...
Lightcurve fetched: 141 alerts, passbands: ['R' 'i' 'g' 'z']


,ant_passband,ant_mjd,ant_mag,ant_magerr
time,,,,
2026-02-17 02:21:11.105457,R,61088.098045,23.723501,0.143402
2026-02-19 01:17:06.229662,i,61090.053544,23.417507,0.159649
2026-02-19 01:18:20.914365,i,61090.054409,23.255661,0.149965
2026-02-19 01:20:13.104719,i,61090.055707,23.507910,0.189936
2026-02-19 01:23:21.503963,i,61090.057888,23.510099,0.184778


Extracted lightcurve features for theorized lightcurve in 8.22s!
Engineering features...

Extracted feature row shape: (82, 84)


## 5. Feature alignment check

This is the critical validation step. We compare:
- The columns in `df_extracted` (what reLAISS gave us)
- The columns your models were trained on

**Missing features** = reLAISS didn't compute them for this LSST object (may need to be NaN-filled)  
**Extra features** = reLAISS returned something the model doesn't use (safe to ignore)

In [12]:
extracted_cols = set(df_extracted.columns)

# -- Compare against the full feature set from constants (USE_HOST=True, drop *_err) --
# This mirrors exactly what both training notebooks did with the overlap() function
all_training_lc   = [c for c in default_lc_features   if not c.endswith("_err")]
all_training_host = [c for c in default_host_features if not c.endswith("_err")]
all_training_feats = all_training_lc + all_training_host

missing_from_extraction = [c for c in all_training_feats if c not in extracted_cols]
extra_in_extraction     = [c for c in extracted_cols if c not in all_training_feats
                           and c not in ("ztf_object_id", "lsst_dia_object_id")]

print("=" * 60)
print(f"Training features expected:   {len(all_training_feats)}")
print(f"Features in extracted output: {len(extracted_cols)}")
print()
print(f"MISSING from extraction ({len(missing_from_extraction)}):")
for c in missing_from_extraction:
    print(f"  - {c}")
print()
print(f"EXTRA in extraction (not in training, {len(extra_in_extraction)}):")
for c in extra_in_extraction:
    print(f"  + {c}")
print("=" * 60)

Training features expected:   43
Features in extracted output: 84

MISSING from extraction (18):
  - gKronMagCorrected
  - gKronRad
  - gExtNSigma
  - rKronMagCorrected
  - rKronRad
  - rExtNSigma
  - iKronMagCorrected
  - iKronRad
  - iExtNSigma
  - zKronMagCorrected
  - zKronRad
  - zExtNSigma
  - gminusrKronMag
  - rminusiKronMag
  - iminuszKronMag
  - rmomentXX
  - rmomentXY
  - rmomentYY

EXTRA in extraction (not in training, 57):
  + g_peak_mag_err
  + mjd_cutoff
  + r_n_peaks_err
  + g_beyond_2sigma_err
  + g_n_peaks_err
  + r_decline_time_err
  + g_duration_above_half_flux_err
  + features_valid_err
  + r_decline_local_curvature_err
  + g_mean_rolling_variance_err
  + g_peak_mag
  + r_secondary_peak_width_err
  + r_peak_time_err
  + r_secondary_peak_prominence
  + g_decline_local_curvature_err
  + r_duration_above_half_flux_err
  + r_dmag_secondary_peak
  + g_secondary_peak_prominence
  + g_dmag_secondary_peak_err
  + r_beyond_2sigma_err
  + r_rise_local_curvature_err
  + g_dec

In [13]:

if iso_feature_cols is not None:
    iso_missing = [c for c in iso_feature_cols if c not in extracted_cols]
    print(f"IsoForest: {len(iso_missing)} features missing from extraction:")
    print(iso_missing if iso_missing else "  None -- all present!")
    print()

if dimmad_feature_cols is not None:
    dim_missing = [c for c in dimmad_feature_cols if c not in extracted_cols]
    print(f"DiMMAD: {len(dim_missing)} features missing from extraction:")
    print(dim_missing if dim_missing else "  None -- all present!")

IsoForest: 0 features missing from extraction:
  None -- all present!

DiMMAD: 0 features missing from extraction:
  None -- all present!


## 6. Inspect the extracted values

Check for NaNs in the training features — too many NaNs in a single row may
cause the KNNImputer to struggle, since it was fit on the training set distribution.

In [15]:
from relaiss import constants

lc_features_no_err = [c for c in constants.lc_features_const if not c.endswith("_err")]
missing_lc = [c for c in lc_features_no_err if c not in df_extracted.columns]
present_lc = [c for c in lc_features_no_err if c in df_extracted.columns]

print(f"LC features present: {len(present_lc)} / {len(lc_features_no_err)}")
print(f"Missing LC features: {missing_lc if missing_lc else 'None -- all present!'}")

LC features present: 25 / 25
Missing LC features: None -- all present!


In [16]:
# Subset extracted df to just the columns the models care about
present_feat_cols = [c for c in all_training_feats if c in extracted_cols]
df_feat_subset = df_extracted[present_feat_cols].copy()

nan_counts = df_feat_subset.isna().sum(axis=1)
print(f"NaN count in extracted feature row: {nan_counts.values[0]} / {len(present_feat_cols)} features")
print(f"({nan_counts.values[0]/len(present_feat_cols)*100:.1f}% missing)")

# Show which features are NaN
nan_cols = df_feat_subset.columns[df_feat_subset.isna().any()].tolist()
if nan_cols:
    print(f"\nFeatures with NaN values ({len(nan_cols)}):")
    for c in nan_cols:
        print(f"  {c}")
else:
    print("\nNo NaN values -- great!")

# Display extracted values for all features
display(df_feat_subset.T.rename(columns={df_feat_subset.index[0]: "extracted_value"}))

NaN count in extracted feature row: 0 / 25 features
(0.0% missing)

No NaN values -- great!


,extracted_value,1,2,3,4,5,6,7,8,9,...,72,73,74,75,76,77,78,79,80,81
g_peak_time,105.009800,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,...,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02,1.050098e+02
r_peak_time,6.008776,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,...,8.016141e+00,8.016141e+00,9.995432e+00,9.998044e+00,9.998044e+00,9.998044e+00,9.998044e+00,9.998044e+00,9.998044e+00,9.998044e+00
g_rise_time,71.026893,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,...,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01,7.102689e+01
g_decline_time,60.757580,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,...,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01,6.075758e+01
r_rise_time,6.008776,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,6.008776e+00,...,7.287798e+00,7.287798e+00,6.983283e+00,7.991403e+00,7.991403e+00,7.991403e+00,7.991403e+00,7.991403e+00,7.991403e+00,7.991403e+00
r_decline_time,0.008742,8.741548e-03,8.741548e-03,8.741548e-03,8.741548e-03,9.760759e-01,9.765096e-01,9.769433e-01,9.778127e-01,9.795482e-01,...,1.974233e+00,1.975964e+00,6.210983e+01,6.210983e+01,3.051684e-03,3.490446e-03,4.006399e-03,7.478552e-03,7.478552e-03,7.478552e-03
g_duration_above_half_flux,6.959105,6.961742e+00,6.970216e+00,6.971954e+00,6.978171e+00,6.984852e+00,6.985286e+00,6.985720e+00,6.986589e+00,6.988325e+00,...,9.990374e+00,9.992105e+00,9.995432e+00,9.998044e+00,1.000110e+01,1.000153e+01,1.000205e+01,1.000552e+01,1.094623e+01,1.699516e+01
r_duration_above_half_flux,6.959105,6.961742e+00,6.970216e+00,6.971954e+00,6.978171e+00,6.984852e+00,6.985286e+00,6.985720e+00,6.986589e+00,6.988325e+00,...,9.990374e+00,9.992105e+00,9.995432e+00,9.998044e+00,1.000110e+01,1.000153e+01,1.000205e+01,1.000552e+01,1.094623e+01,1.699516e+01
g_amplitude,0.000000,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,...,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02,1.323427e-02
r_amplitude,0.551403,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,5.514028e-01,...,8.651436e-01,8.651436e-01,8.841701e-01,9.375529e-01,9.375529e-01,9.375529e-01,9.375529e-01,9.375529e-01,9.375529e-01,9.375529e-01


## 8. Batch processing template

For full list of LSST object IDs, use to loop over and collect all features in one DataFrame.

In [18]:
import os
os.getcwd()

'/Users/jennakempster-taylor/re-laiss'

In [23]:
from tqdm import tqdm
import pandas as pd
from pathlib import Path

base_dir = Path("runs/lsst_run_2026_03_16")
base_dir.mkdir(parents=True, exist_ok=True)

# Load your source list
sources = pd.read_csv("data/tns_search.csv")
lsst_ids = sources["ID"].astype(str).tolist()
print(f"Total objects to process: {len(lsst_ids)}")

checkpoint_path = Path("lsst_extracted_features.csv")
failed = []

# Check what's already been done
already_done = []
if checkpoint_path.exists():
    done_df = pd.read_csv(checkpoint_path)
    already_done = done_df["lsst_dia_object_id"].astype(str).tolist()
    print(f"Resuming — {len(already_done)} already extracted, {len(lsst_ids) - len(already_done)} remaining")

for lsst_id in tqdm(lsst_ids, desc="Extracting features"):
    if lsst_id in already_done:
        continue
    try:
        df_row = extract_features_for_lsst_object(lsst_id)
        # Append to CSV incrementally so progress is never lost
        df_row.to_csv(checkpoint_path, mode="a",
                      header=not checkpoint_path.exists(), index=False)
        already_done.append(lsst_id)
    except Exception as e:
        print(f"FAILED: {lsst_id} — {e}")
        failed.append({"lsst_id": lsst_id, "error": str(e)})

print(f"\nDone. Extracted: {len(already_done)} / {len(lsst_ids)}")
print(f"Failed: {len(failed)}")
if failed:
    pd.DataFrame(failed).to_csv("lsst_failed.csv", index=False)
    print("Failed IDs saved to lsst_failed.csv")


Total objects to process: 50
Resuming — 221 already extracted, -171 remaining


Extracting features:   0%|                               | 0/50 [00:00<?, ?it/s]

Fetching LSST 203221 from Antares...


Extracting features:   2%|▍                      | 1/50 [00:00<00:08,  6.00it/s]

FAILED: 203221 — Object 203221 not found in Antares
Fetching LSST 203198 from Antares...


...

FAILED: 202561 — Object 202561 not found in Antares
Fetching LSST 202560 from Antares...
FAILED: 202560 — Object 202560 not found in Antares
Fetching LSST 202549 from Antares...


Extracting features:  96%|█████████████████████ | 48/50 [00:08<00:00,  6.03it/s]

FAILED: 202549 — Object 202549 not found in Antares
Fetching LSST 202548 from Antares...
FAILED: 202548 — Object 202548 not found in Antares
Fetching LSST 202514 from Antares...


Extracting features: 100%|██████████████████████| 50/50 [00:09<00:00,  5.40it/s]

FAILED: 202514 — Object 202514 not found in Antares
Fetching LSST 202511 from Antares...
FAILED: 202511 — Object 202511 not found in Antares

Done. Extracted: 221 / 50
Failed: 50
Failed IDs saved to lsst_failed.csv


In [25]:
### df_all = pd.read_csv("lsst_extracted_features.csv")
print(f"Rows in CSV: {len(df_all)}")
print(f"Unique objects: {df_all['lsst_dia_object_id'].nunique()}")

NameError: name 'df_all' is not defined

In [27]:
df_all = pd.read_csv("lsst_extracted_features.csv")
df_all = df_all.drop_duplicates(subset="lsst_dia_object_id")
df_all.to_csv("lsst_extracted_features.csv", index=False)
print(f"Cleaned CSV: {len(df_all)} rows, {df_all['lsst_dia_object_id'].nunique()} unique objects")

Cleaned CSV: 221 rows, 221 unique objects
